# EcoEye — Bird Image Scraper

Scrapes bird images from iStock using Selenium with paginated navigation.  
Part of the EcoEye Online Bird Monitoring System — Loyalist College AI & Data Science Program.

**How to use:**
1. Install dependencies: `pip install -r requirements.txt`
2. Download ChromeDriver matching your Chrome version from https://chromedriver.chromium.org/
3. Update `CHROMEDRIVER_PATH` and `SPECIES_NAME` in the Configuration cell below
4. Run all cells

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import requests
from datetime import datetime
import os
import time

In [ ]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
# Update these values before running

SPECIES_NAME = "myna"  # Bird species to search for on iStock
START_PAGE   = 1       # First page to scrape
END_PAGE     = 5       # Last page to scrape (adjust as needed)

# Path to your ChromeDriver executable
# Download from: https://chromedriver.chromium.org/
CHROMEDRIVER_PATH = "chromedriver.exe"  # Place chromedriver.exe in this folder, or provide full path

# Images will be saved to: ./downloaded_images/<SPECIES_NAME>/
SAVE_FOLDER = os.path.join(os.getcwd(), "downloaded_images", SPECIES_NAME)
os.makedirs(SAVE_FOLDER, exist_ok=True)

print(f"Saving images to: {SAVE_FOLDER}")

In [ ]:
def generate_image_name(species):
    """Generate a unique filename using species name and current timestamp."""
    timestamp = datetime.now().strftime("%d-%m-%Y-%H-%M-%S-%f")
    return f"{species}_{timestamp}.jpg"


def setup_driver(driver_path):
    """Initialize and return a Chrome WebDriver instance."""
    service = Service(driver_path)
    driver = webdriver.Chrome(service=service)
    return driver


def scrape_images(driver, url):
    """Navigate to a URL and return all image src URLs found on the page."""
    driver.get(url)
    time.sleep(5)  # Wait for dynamic content to load
    image_elements = driver.find_elements(By.TAG_NAME, "img")
    image_urls = [
        img.get_attribute("src")
        for img in image_elements
        if img.get_attribute("src")
    ]
    return image_urls


def download_images(image_urls, save_folder, species):
    """Download and save images from a list of URLs."""
    downloaded = 0
    for image_url in image_urls:
        response = requests.get(image_url)
        if response.status_code == 200:
            image_name = generate_image_name(species)
            save_path = os.path.join(save_folder, image_name)
            with open(save_path, "wb") as f:
                f.write(response.content)
            downloaded += 1
            print(f"  Saved: {image_name}")
        else:
            print(f"  Failed ({response.status_code}): {image_url}")
    return downloaded


def main(species, base_url, start_page, end_page, driver_path, save_folder):
    """Scrape images across paginated results and save them locally."""
    driver = setup_driver(driver_path)
    total_downloaded = 0

    try:
        for page in range(start_page, end_page + 1):
            url = base_url.format(page=page)
            print(f"\nScraping page {page}: {url}")
            images = scrape_images(driver, url)
            count = download_images(images, save_folder, species)
            total_downloaded += count
            print(f"  → {count} images downloaded from page {page}")
    finally:
        driver.quit()

    print(f"\nDone. Total images downloaded: {total_downloaded}")
    print(f"Saved to: {save_folder}")

In [ ]:
# ─── RUN ──────────────────────────────────────────────────────────────────────
BASE_URL = "https://www.istockphoto.com/search/2/image-film?istockcollection=&mood=&page={page}&phrase=" + SPECIES_NAME

main(
    species=SPECIES_NAME,
    base_url=BASE_URL,
    start_page=START_PAGE,
    end_page=END_PAGE,
    driver_path=CHROMEDRIVER_PATH,
    save_folder=SAVE_FOLDER
)